# Análisis Avanzado de Datos & Modelado
**Institución:** Universidad Católica Boliviana "San Pablo"  
**Programa:** Maestría en Ciencia de Datos  
**Autor:** Carlos Andrés Pérez Guzmán  
**Fecha:** Junio 2026  

---
### ⚙️ Entorno y Objetivos del Notebook
> **Nota:** Este cuaderno implementa una prueba de concepto para el proyecto de segunda fase en laMaestrái de Ciencia de Datos.

* **Fase del Proyecto:** Prueba de concepto
* **Core Tech Stack:** `Python` (Programación general para ciencia de datos)
---

<h3><span style="color:#05856d">1. Librerías</span></h3>

---

In [1]:
# Manupulación de datos y cálculos
import polars as pl
import polars.selectors as cs

<h3><span style="color:#05856d">2. Carga y limpieza de información</span></h3>

---

In [2]:
from s_01_functions import load_data_own, limpiar_texto_own

df = (
    load_data_own('iowa_liquor_sales_2026_1263_rows.csv')

    .with_columns(
        limpiar_texto_own(column_param = cs.string() - cs.by_name('ordered_on')),
        pl.col('ordered_on').str.to_date(format = '%Y-%m-%d')
        )
    .with_columns(
        pl.col('im_desc','category_name').cast(dtype = pl.Categorical())
        )

    )

df.glimpse()

📦 Iniciando carga de datos locales desde: iowa_liquor_sales_2026_1263_rows.csv
✅ Carga completada: 964947 registros en 0.61 segundos.
Rows: 964947
Columns: 21
$ bottle_volume_ml  <f64> 750.0, 1750.0, 375.0, 50.0, 1750.0, 375.0, 375.0, 750.0, 750.0, 50.0
$ category_code     <i64> 1011100, 1012100, 1012100, 1012100, 1062400, 1062400, 1011200, 1011200, 1011200, 1011200
$ category_name     <cat> BLENDED WHISKIES, CANADIAN WHISKIES, CANADIAN WHISKIES, CANADIAN WHISKIES, SPICED RUM, SPICED RUM, STRAIGHT BOURBON WHISKIES, STRAIGHT BOURBON WHISKIES, STRAIGHT BOURBON WHISKIES, STRAIGHT BOURBON WHISKIES
$ county_fips_code  <i64> 19177, 19177, 19177, 19177, 19177, 19177, 19177, 19177, 19177, 19177
$ county_name       <str> 'VAN BUREN', 'VAN BUREN', 'VAN BUREN', 'VAN BUREN', 'VAN BUREN', 'VAN BUREN', 'VAN BUREN', 'VAN BUREN', 'VAN BUREN', 'VAN BUREN'
$ im_desc           <cat> RED STAG BLACK CHERRY, BLACK VELVET, BLACK VELVET, CROWN ROYAL MINI, ADMIRAL NELSON SPICED, ADMIRAL NELSON SPICED, JIM BEAM

<h3><span style="color:#05856d">3. Selección de variables</span></h3>

---

Es importante identificar qué variables tienen alguna funcionalidad estratégica al construir un modelo de decisión

In [3]:
# Variables del dataset completo
df_all_varaibles = df.columns

# Variables con utilidad estratégica
df_selected_variables = [
    'ordered_on','item_no','im_desc','category_name','sales_bottles',
    'sales_dollars','pack','bottle_volume_ml']

# Variables sin utilidad estratégica
df_eliminated_variables = set(df_all_varaibles) - set(df_selected_variables)
print("Variables sin utilidad estratégica:\n", *sorted(df_eliminated_variables), sep='\n')

Variables sin utilidad estratégica:

category_code
county_fips_code
county_name
invoice_id
sales_gallons
sales_liters
store_address
store_city
store_name
store_no
store_zip_code
vendor_name
vendor_number


In [4]:
# Seleccionar variables de interés
df_01 = df.select(df_selected_variables)
df_01.glimpse()

# Eliminar varaibles intermedias
del df

Rows: 964947
Columns: 8
$ ordered_on       <date> 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18
$ item_no           <i64> 27544, 11788, 11774, 11290, 43028, 43024, 19064, 27783, 27410, 19061
$ im_desc           <cat> RED STAG BLACK CHERRY, BLACK VELVET, BLACK VELVET, CROWN ROYAL MINI, ADMIRAL NELSON SPICED, ADMIRAL NELSON SPICED, JIM BEAM, JIM BEAM APPLE, JIM BEAM HONEY, JIM BEAM MINI
$ category_name     <cat> BLENDED WHISKIES, CANADIAN WHISKIES, CANADIAN WHISKIES, CANADIAN WHISKIES, SPICED RUM, SPICED RUM, STRAIGHT BOURBON WHISKIES, STRAIGHT BOURBON WHISKIES, STRAIGHT BOURBON WHISKIES, STRAIGHT BOURBON WHISKIES
$ sales_bottles     <f64> 2.0, 6.0, 7.0, 2.0, 12.0, 8.0, 7.0, 2.0, 2.0, 2.0
$ sales_dollars     <f64> 34.5, 103.5, 34.16, 17.28, 225.0, 42.0, 68.25, 34.5, 34.5, 21.0
$ pack              <i64> 12, 6, 24, 10, 6, 24, 24, 12, 12, 12
$ bottle_volume_ml  <f64> 750.0, 1750.0, 375.0, 50.0, 1750.0, 375.0, 375.0, 75

La llave que le da identidad a un producto es la combinación de item_no y im_desc

In [5]:
df_01.select('item_no','im_desc').unique().select('im_desc').is_duplicated().any()

True

In [6]:
df_01.select('item_no','im_desc').unique().select('item_no').is_duplicated().any()

True

In [7]:
df_01.select('item_no','im_desc').unique().is_duplicated().any()

False

¿Existen productos con más de una categoría?

In [8]:
df_01.select('category_name','item_no','im_desc').unique().is_duplicated().any()

False

In [9]:
(
    df_01.select('category_name', 'item_no', 'im_desc')
    .unique()
    # Mantenemos las 3 columnas, pero contamos si el par (item, desc) se repite más de 1 vez
    .filter(pl.len().over('item_no', 'im_desc') > 1)
    .sort('item_no')
)

category_name,item_no,im_desc
cat,i64,cat
"""TEMPORARY & SPECIALTY PACKAGES""",6768,"""LAGAVULIN SWEET PEAT"""
"""SCOTCH WHISKIES""",6768,"""LAGAVULIN SWEET PEAT"""
"""TEMPORARY & SPECIALTY PACKAGES""",7224,"""MACALLAN SHERRY OAK 12YO 110 P…"
"""SINGLE MALT SCOTCH""",7224,"""MACALLAN SHERRY OAK 12YO 110 P…"
"""IRISH WHISKIES""",14812,"""JAMESON ORIGINAL MLS SPECIAL E…"
…,…,…
"""TEMPORARY & SPECIALTY PACKAGES""",73985,"""EVAN WILLIAMS BLACKBERRY MINI"""
"""TEMPORARY & SPECIALTY PACKAGES""",80276,"""JACKSON MORGAN SOUTHERN CREAM …"
"""CREAM LIQUEURS""",80276,"""JACKSON MORGAN SOUTHERN CREAM …"


<h3><span style="color:#05856d">4. Análisis exploratorio</span></h3>

---

¿Qué producto vendió más botellas?

In [10]:
analysis_01 = (
    df_01
    .group_by('item_no','im_desc')
    .agg(pl.sum('sales_bottles').alias('count'))
    .sort( by = 'count', descending = True)
    )

print(f"Existen {analysis_01.n_unique(subset = ['item_no','im_desc'])} productos en el dataset.")
analysis_01.show()

Existen 4048 productos en el dataset.
shape: (5, 3)
┌─────────┬─────────────────────────────────┬──────────┐
│ item_no ┆ im_desc                         ┆ count    │
│ ---     ┆ ---                             ┆ ---      │
│ i64     ┆ cat                             ┆ f64      │
╞═════════╪═════════════════════════════════╪══════════╡
│ 64870   ┆ FIREBALL CINNAMON WHISKEY       ┆ 953182.0 │
│ 65013   ┆ FIREBALL CINNAMON WHISKEY MINI… ┆ 361013.0 │
│ 36904   ┆ MCCORMICK 80PRF VODKA PET       ┆ 252071.0 │
│ 38177   ┆ TITOS HANDMADE VODKA            ┆ 195662.0 │
│ 84171   ┆ 99 BANANAS                      ┆ 188879.0 │
└─────────┴─────────────────────────────────┴──────────┘


¿Qué categoría generó más ganacias?

In [11]:
analysis_02 = (
    df_01
    .group_by('category_name')
    .agg( (pl.sum('sales_dollars')/1e6).alias('count_M') )
    .sort( by = 'count_M', descending = True)
    )

print(f"Existen {analysis_02.n_unique(subset = 'category_name')} categorías únicas en el dataset.")
analysis_02.show()

Existen 44 categorías únicas en el dataset.
shape: (5, 2)
┌───────────────────────────┬───────────┐
│ category_name             ┆ count_M   │
│ ---                       ┆ ---       │
│ cat                       ┆ f64       │
╞═══════════════════════════╪═══════════╡
│ AMERICAN VODKAS           ┆ 25.500022 │
│ CANADIAN WHISKIES         ┆ 17.243223 │
│ STRAIGHT BOURBON WHISKIES ┆ 15.29209  │
│ 100% AGAVE TEQUILA        ┆ 12.68607  │
│ WHISKEY LIQUEUR           ┆ 10.740141 │
└───────────────────────────┴───────────┘


In [12]:
analysis_03 = (
    df_01
    .group_by(['item_no', 'im_desc'])
    .agg(
        # 1. ¿Cuántas botellas se han vendido en la historia de este producto?
        pl.sum('sales_bottles').alias('volumen_total'),
        
        # 2. ¿En cuántos días únicos alguien compró al menos 1 botella?
        pl.n_unique('ordered_on').alias('frecuencia_dias_demanda')
    )
    # Remover elementos raros (productos que quizá se venden mucho pero solo en 2 días del año)
    .filter(pl.col("frecuencia_dias_demanda") > 100)
    
    # Ordenamos por los que tienen demanda más constante, y luego por volumen
    .sort(["frecuencia_dias_demanda", "volumen_total"], descending=[True, True])
)

# Vemos el Top 10 de candidatos perfectos para el agente
analysis_03.show(10)

shape: (10, 4)
┌─────────┬─────────────────────────────────┬───────────────┬─────────────────────────┐
│ item_no ┆ im_desc                         ┆ volumen_total ┆ frecuencia_dias_demanda │
│ ---     ┆ ---                             ┆ ---           ┆ ---                     │
│ i64     ┆ cat                             ┆ f64           ┆ u32                     │
╞═════════╪═════════════════════════════════╪═══════════════╪═════════════════════════╡
│ 36904   ┆ MCCORMICK 80PRF VODKA PET       ┆ 252071.0      ┆ 115                     │
│ 38177   ┆ TITOS HANDMADE VODKA            ┆ 195662.0      ┆ 115                     │
│ 11788   ┆ BLACK VELVET                    ┆ 159974.0      ┆ 115                     │
│ 48254   ┆ HENNESSY VS FLASK               ┆ 35671.0       ┆ 115                     │
│ 34456   ┆ KETEL ONE                       ┆ 11455.0       ┆ 115                     │
│ 64870   ┆ FIREBALL CINNAMON WHISKEY       ┆ 953182.0      ┆ 114                     │
│ 65013   ┆ FIREB

<h3><span style="color:#05856d">5. Resampling e imputación de 0's</span></h3>

---

El seleccionado para la simulación es MCCORMICK 80PRF VODKA PET (Item No: 36904). Se ha vendido en 115 días distintos con un volumen brutal de más de 250,000 botellas. Entonces, se aislará McCormick, luego se sumarán todas sus ventas por día (porque en un día puede haber sido comprado por 50 tiendas distintas) y, lo más crítico: se crearán los días en los que no se vendió nada para colcarles Demanda = 0.

In [13]:
# 1. Filtrar y agrupar el producto tipo
analysis_04 = (
    df_01
    .filter((pl.col('item_no') == 36904) & 
            (pl.col('im_desc') == 'MCCORMICK 80PRF VODKA PET') )
    .group_by('ordered_on')
    .agg(
        pl.sum('sales_bottles').alias('demanda_diaria'),
        pl.sum('sales_dollars').alias('ingresos_diarios') )
)

# 2 Imputación estocástica de ceros (0's)

aux_calendario = pl.DataFrame({
    'ordered_on': pl.date_range(
        start = pl.date(year = 2026, month = 1, day = 1),
        end = analysis_04['ordered_on'].max(),
        interval = '1d',
        eager = True)
})

analysis_05 = (
    aux_calendario
    .join(analysis_04, on = 'ordered_on', how = 'left', validate = 'm:1')
    .with_columns([
        # Si ese día no hubo ventas, la demanda y el ingreso son 0
        pl.col('demanda_diaria').fill_null(0),
        pl.col('ingresos_diarios').fill_null(0.0)
    ])
)

print(f"Días totales de simulación: {analysis_05.height}")
analysis_05.head(10)

Días totales de simulación: 150


ordered_on,demanda_diaria,ingresos_diarios
date,f64,f64
2026-01-01,0.0,0.0
2026-01-02,1741.0,4782.0
2026-01-03,0.0,0.0
2026-01-04,576.0,1584.0
2026-01-05,3031.0,8332.0
2026-01-06,2201.0,6050.0
2026-01-07,1082.0,2971.0
2026-01-08,1588.0,4365.0
2026-01-09,2400.0,6596.0


Si se estudia el resultado, es probable observar algunos días con demanda_diaria = 0. Si no hiciéramos este paso de rellenar los ceros, el agente de AI (Deep Reinforcement Learning) creería que todos los días se vende producto. Al poner los ceros, el agente aprenderá por las malas que si pide mucho inventario en un día malo, se quedará con el producto estancado y se le cobrarán costos de almacenamiento (Holding Costs).

<h3><span style="color:#05856d">6. Parámetros del entorno (MDP Constantes)</span></h3>

---

Una vez generada la serie de tiempo continua, se deben definir las constantes físicas y financieras que regirán la simulación logística:

- **Precio de Venta ($p$)**: Calculado dinámicamente como $\sum Ingresos / \sum Demanda$. Representa el flujo de caja de entrada.

- **Tamaño del Lote ($pack$)**: Extraído del registro original del producto seleccionado. El agente no pedirá botellas sueltas, pedirá "cajas". Su acción $A_t = 1$ sumará el valor de un $pack$ al inventario.

- **Costo de Mantenimiento / Holding Cost ($h$)**: Penalización impuesta por cada unidad de producto no vendida que permanece en el almacén al final del día.

- **Costo por Quiebre de Stock / Stockout Cost ($k$)**: Penalización aplicada al agente si la demanda es mayor a su inventario (ventas perdidas / pérdida de buena voluntad del cliente).

- **Lead Time ($L$)**: El retraso de tiempo entre que el agente decide hacer un pedido a la destilería y el momento en que las botellas llegan físicamente al almacén para poder ser despachadas.

In [14]:
df_01.glimpse()

Rows: 964947
Columns: 8
$ ordered_on       <date> 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18
$ item_no           <i64> 27544, 11788, 11774, 11290, 43028, 43024, 19064, 27783, 27410, 19061
$ im_desc           <cat> RED STAG BLACK CHERRY, BLACK VELVET, BLACK VELVET, CROWN ROYAL MINI, ADMIRAL NELSON SPICED, ADMIRAL NELSON SPICED, JIM BEAM, JIM BEAM APPLE, JIM BEAM HONEY, JIM BEAM MINI
$ category_name     <cat> BLENDED WHISKIES, CANADIAN WHISKIES, CANADIAN WHISKIES, CANADIAN WHISKIES, SPICED RUM, SPICED RUM, STRAIGHT BOURBON WHISKIES, STRAIGHT BOURBON WHISKIES, STRAIGHT BOURBON WHISKIES, STRAIGHT BOURBON WHISKIES
$ sales_bottles     <f64> 2.0, 6.0, 7.0, 2.0, 12.0, 8.0, 7.0, 2.0, 2.0, 2.0
$ sales_dollars     <f64> 34.5, 103.5, 34.16, 17.28, 225.0, 42.0, 68.25, 34.5, 34.5, 21.0
$ pack              <i64> 12, 6, 24, 10, 6, 24, 24, 12, 12, 12
$ bottle_volume_ml  <f64> 750.0, 1750.0, 375.0, 50.0, 1750.0, 375.0, 375.0, 75

In [15]:
aux_parametros_entorno = (
    df_01
    .filter((pl.col('item_no') == 36904) & 
            (pl.col('im_desc') == 'MCCORMICK 80PRF VODKA PET') )
    .select([

        # Precio de venta promedio por botella
        ( (pl.col('sales_dollars').sum()) / (pl.col('sales_bottles').sum()) ).alias('precio_unitario_promedio'),
        
        # Tamaño de la caja (pack) - usualmente es el mismo para todo el producto
        pl.col('pack').first().alias('unidades_por_caja')
    ])
)

print(aux_parametros_entorno)

shape: (1, 2)
┌──────────────────────────┬───────────────────┐
│ precio_unitario_promedio ┆ unidades_por_caja │
│ ---                      ┆ ---               │
│ f64                      ┆ i64               │
╞══════════════════════════╪═══════════════════╡
│ 2.757891                 ┆ 24                │
└──────────────────────────┴───────────────────┘
